# PROJECT 2: US National Housing Market Analysis & price prediction
# Business goal: Predict Median Sales price of house sold using economic & housing supply/demand indicators (Monthly us data 1990-2025)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score

# Load data
df = pd.read_csv('Data/unified_monthly_data_interpolated_1990_20250101.csv')
print(f"Dataset shape: {df.shape}")
print("Columns:", df.columns.tolist())
print("\nFirst 5 rows:")
print(df.head())

# Convert Date and Extract simple time features(Year/month)
df['Date'] = pd.to_datetime(df['Date'])
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month

# Drop non predictive columns
df = df.drop(columns=['Date', 'Region'])

In [ ]:
# 2. Quick EDA

print(df.describe().round(2))

# Target variable (What we want to predict)
wayne_target = 'MedianSalesPriceofHousesSold'

# Target Distribution
plt.figure(figsize=(10, 6))
sns.histplot(data=df, x=wayne_target, kde=True, color='skyblue') # series fix of argument any i.e from (df[wayne_target]) to(data=df, x=wayne_target) 
plt.title('Target Distribution: Median Sales price of House Sold ($)')
plt.xlabel('Price ($)')
plt.ylabel('Frequency')
plt.grid(True, alpha=0.3)
plt.show()

# Correlation heatmap (all numeric features)
plt.figure(figsize=(14, 10))
corr_matrix = df.corr()
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', center=0, linewidths=0.5)
plt.title('Correlation Heatmap of All Housing & Economic Features')
plt.tight_layout()
plt.show()

In [ ]:
# 3. Simple Baseline (Always predict the mean.)
baseline = df[wayne_target].mean()
baseline_mae = mean_absolute_error(df[wayne_target], [baseline] * len(df))
print(f"Baseline MAE (predict mean price): ${baseline_mae:,.2f}")

In [ ]:
# 4. FEATURE ENGINEERING & MODEL
X = df.drop(wayne_target, axis=1)
y = df[wayne_target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

bruce_model = Ridge(alpha=0.1)
bruce_model.fit(X_train, y_train)

preds = bruce_model.predict(X_test)

mae = mean_absolute_error(y_test, preds)
r2 = r2_score(y_test, preds)

print(f"Ridge Regression MAE: ${mae:,.2f}")
print(f"R2 Score: {r2:.4f} (higher = better fit)")

In [ ]:
# 5. Feature Importance (RIDGE COEFFICIENTS)
coef_series = pd.Series(bruce_model.coef_, index=X.columns).sort_values(ascending=False)

print("Top 10 most influential features (coefficient values):")
print(coef_series.head(10).round(2))

# Plot feature importance.
plt.figure(figsize=(10, 8))
coef_series.plot(kind='barh', color='teal')
plt.title('Feature Importance (Ridge Model Coefficients)')
plt.xlabel('Coefficients (impact on predicted price)')
plt.ylabel('Feature')
plt.gca().invert_yaxis()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Business recommendations
print("\n6. Business recommendations")
print("""
1. Home Price Index, Average Sales Price, and Unemployment Rate are the strongest drivers.
      Action: Monitor these 3 indicators closely for accurate price forecasting.
2. Vacant Housing units and supply metrics (e.g monthlysupplyofnewhouses ) show strong negative impact.
      Action: increased housing supply can help moderate price growth.
3. Model (r2, -0.95+) far outperforms naive baseline to highly usable for forecasting.
      Next Steps:
        . add time series models (arima/prophet) for future months
        . Deploy as monthly price predictor dashboard

4. Value for insurers / banks / policymakers: Early warning of bubbles or cooling markets. 
""")